# Lab 6: OpenAlex requests and full text

**COMPSS 211A | Fall 2026 | Student copy**

Choose an institution and a year. Get its 25 most-cited works, make a table, and download one paper.

Duplicate this notebook and add `_mine` to its name. Write your code in the empty cells and replace the answer placeholders. Keep the last 10 minutes for the quiz on Monday's material.

You will need a free OpenAlex account and API key. File downloads use part of your daily API allowance. Download one PDF for this lab.


## 1. Set up your key

1. Make a free account at [openalex.org](https://openalex.org). Find your API key in your account settings.
2. In VS Code, open **Terminal > New Terminal** in the top folder of the course repository.
3. If you do not have `.env` yet, run `cp .env.example .env`.
4. Open `.env`: `open -e .env` on Mac, or `notepad .env` on Windows.
5. Put your key after `OPENALEX_API_KEY=`, with no spaces or quotes. Save the file.
6. Run `git status`. `.env` must not appear in the list.

Run the supplied code below to load the key. It looks for `.env` in this folder and its parent folders. Never paste your key into a notebook or an AI chat.


In [ ]:
from pathlib import Path
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("OPENALEX_API_KEY", "")
if not key:
    raise ValueError("Add OPENALEX_API_KEY to your .env file, then run this cell again.")

headers = {"Authorization": "Bearer " + key}
print("Key loaded.")


## 2. Ask for works

| Institution | OpenAlex ID |
| --- | --- |
| UC Berkeley | I95457486 |
| UCLA | I161318765 |
| Stanford | I97018004 |
| MIT | I63966007 |
| Harvard | I136199984 |
| Oxford | I40120149 |
| Amsterdam | I887064364 |

Choose an institution and a year.

1. Make a dictionary called `params` with `"filter"`, `"sort"`, `"per_page"`, and `"cursor"`.
   - Example filter: `"institutions.id:I95457486,publication_year:2024"`
   - Sort: `"cited_by_count:desc"`
   - Page size: `25`
   - Cursor: `"*"`
2. Send a GET request to `https://api.openalex.org/works`, with `params=params`, `headers=headers`, and `timeout=30`. Call the response `response`.
3. Print `response.status_code`. Continue only if it is 200.
4. Read the JSON with `data = response.json()`.
5. Save it to `openalex_mine.json` with `json.dump()`, then read it back with `json.load()`.

From now on, use the saved data. Do not send the request again each time you change your table code.

**If the request fails:** for status 429, wait rather than repeat it. You can use the saved course response below for sections 3–5. It contains five Berkeley-affiliated works from 2024, so describe that sample in your answers. Full-text retrieval still needs a working key and connection.

```python
with open("../docs/interactives/data/week06/openalex_page1.json") as f:
    data = json.load(f)
```


## 3. Check the response

1. Print `data["meta"]["count"]`. How many works matched the filter?
2. Print `len(data["results"])`. How many did you receive?
3. Set `work = data["results"][0]`. Display its title: `work["display_name"]`.
4. Find its field: go through `"primary_topic"`, then `"field"`, then `"display_name"`. A missing primary topic is `None`.

Why can the number of matches be larger than the number of works you received?


Your answer:

## 4. Make a table

Start with `rows = []`. Loop over `data["results"]` and append one dictionary per work. Use these columns:

| Column | Value from each work |
| --- | --- |
| `title` | `display_name` |
| `cited_by` | `cited_by_count` |
| `field` | `primary_topic` → `field` → `display_name` |
| `has_pdf` | `has_content` → `pdf` |

If `primary_topic` is `None`, use `None` for the field. For the PDF flag, use `work.get("has_content", {}).get("pdf", False)`; the saved course response may not include this field.

Make `table = pd.DataFrame(rows)`. Display it. Use `value_counts()` to count the works in each field. Also count missing fields with `table["field"].isna().sum()`.


## 5. Check what the table means

Someone calls your results “this institution's 25 most important papers” and says the field counts show what the institution is best at.

In three or four sentences:

- What did your filter and sort actually select?
- Why do citation counts and field counts not establish those claims?
- Why should you say “works with at least one author affiliated with the institution” rather than “papers by its faculty”?

An affiliation record does not tell you whether an author is a faculty member, student, or another kind of researcher.


Your answer:

## 6. Find and download one paper

So imagine you want to read a paper, not just see its title and citation count.

The JSON contains **metadata**: information about a paper. A PDF contains the paper itself. OpenAlex has PDFs for some works, but not all.

First, find one matching paper with a PDF:

1. Copy your settings: `pdf_params = params.copy()`.
2. Add `,has_content.pdf:true` to `pdf_params["filter"]`.
3. Set `pdf_params["per_page"] = 1` and `pdf_params["cursor"] = "*"`.
4. Send a GET request to the works endpoint, with these parameters, `headers=headers`, and `timeout=30`. Call it `pdf_search`.
5. Check the status before reading the JSON into `pdf_data`.
6. Check the length of `pdf_data["results"]`. If it is 0, try another year. If there is a result, set `paper = pdf_data["results"][0]` and display its `display_name`.

This filter changes your sample: it includes only works with an available PDF. It does not measure all papers at the institution.


### Save the PDF

Run the supplied code below. The file endpoint uses the key as a URL parameter. Do not print its request URL or error details: they can contain the key.

For JSON we used `.json()`. For a PDF, use `.content`: the file's bytes. The code checks the status and the PDF file marker before saving.


In [ ]:
pdf_url = paper["content_urls"]["pdf"]

try:
    pdf_response = requests.get(
        pdf_url, params={"api_key": key}, timeout=60
    )
except requests.RequestException:
    print("The download failed. Try again later.")
else:
    print("Status:", pdf_response.status_code)
    if pdf_response.status_code == 200 and pdf_response.content.startswith(b"%PDF-"):
        pdf_path = Path("openalex_paper_mine.pdf")
        pdf_path.write_bytes(pdf_response.content)
        print("Saved:", pdf_path.name)
    else:
        print("No PDF was saved. Check your key and daily allowance.")


## 7. Open and check the full text

Open `openalex_paper_mine.pdf` from your notebook folder.

- Does its title match the OpenAlex record?
- Can you find the paper's main text?
- Give one specific fact from the paper that you could not get from our metadata table.

Downloading a PDF does not extract its text into Python. Text extraction is a separate task. OpenAlex also offers TEI XML for some papers, but we will stop at the PDF in this lab.

[OpenAlex full-text documentation](https://help.openalex.org/access/fulltext/)


Your answer:

## Before you share

Search your notebook for the first few characters of your key with Ctrl+F or ⌘F. The key must not appear in code or output. Do not display `key`, `headers`, or `pdf_response.url`. If you expose a key, rotate it in your account settings.

Your copy should contain your request code, table, written answers, and download status. Keep the PDF for your own inspection; you do not need to commit it.

**Quiz:** keep the last 10 minutes for the three-question quiz on Monday's material.


## Optional extensions

**Read an error.** Copy `params` into `bad_params` and set `"per_page"` to `500`. Send the request with your authorization header. Print its status and response text. What setting must you change?

**Get the next page.** Set `params["cursor"]` to `data["meta"]["next_cursor"]`. If it is not `None`, request the next page and add its works to your table. Does the field count change? Use the saved response if no more requests are available.
